# NLP Text Processing Pipeline & Feature Engineering

This notebook implements the full pipeline from Data Preprocessing to Feature Engineering and Sentiment Classification.

## Task 1: Preprocessing
1. Convert text to lowercase
2. Tokenization
3. Remove punctuation
4. Remove stopwords
5. Lemmatization

In [5]:

!pip install -y chromium-chromedriver
!pip install selenium pandas


[optparse.groups]Usage:[/]   
  pip install \[options] <requirement specifier> \[package-index-options] ...
  pip install \[options] -r <requirements file> \[package-index-options] ...
  pip install \[options] [-e] <vcs project url> ...
  pip install \[options] [-e] <local project path> ...
  pip install \[options] <archive url/path> ...

no such option: -y


In [6]:
!pip install selenium webdriver-manager

In [ ]:
import time
import random
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--start-maximized")
options.add_argument(r"user-data-dir=C:\temp\selenium_profile")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
# 🟢 Step 1: Open Amazon manually (IMPORTANT)
driver.get("https://www.amazon.in/")
input("👉 Solve CAPTCHA/login manually if shown, then press ENTER...")

# 🔗 Step 2: Product ASIN
ASIN = "B0FLJY793G"

reviews = []
reviews = []

for page in range(1, 25):
    url = f"https://www.amazon.in/product-reviews/{ASIN}/?pageNumber={page}&sortBy=recent"
    
    driver.get(url)
    time.sleep(random.uniform(5, 8))

    print("Current URL:", driver.current_url)

    elems = driver.find_elements(By.CSS_SELECTOR, "span[data-hook='review-body']")
    
    print(f"Page {page} reviews:", len(elems))

    for e in elems:
        text = e.text.strip()
        if text:
            reviews.append(text)

# 🔁 Step 3: Loop pages (direct URL navigation — NO next button)
"""for page in range(1, 100):
    print(f"Scraping page {page}")

    url = f"https://www.amazon.in/product-reviews/{ASIN}/?pageNumber={page}&reviewerType=all_reviews&sortBy=recent"
    driver.get(url)

    time.sleep(random.uniform(5, 8))  # human delay

    # Scroll (simulate user)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    time.sleep(3)
    #time.sleep(random.uniform(2, 4))

    elems = driver.find_elements(By.CSS_SELECTOR, "span[data-hook='review-body']")

    # If blocked
    if len(elems) == 0:
        print("⚠️ No reviews found (blocked/CAPTCHA). Waiting...")
        time.sleep(10)
        continue

    for e in elems:
        text = e.text.strip()
        text = re.sub(r'\s+', ' ', text)

        if text and text not in reviews:
            reviews.append(text)
"""
    if len(reviews) >= 200:
        break

driver.quit()

# 📊 Step 4: Save ONLY review_text
df = pd.DataFrame({"review_text": reviews[:200]})
df.to_csv("amazon_reviews_200.csv", index=False, encoding="utf-8")

print(f"✅ SUCCESS: Saved {len(df)} reviews")

Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5
Scraping page 6
Scraping page 7
Scraping page 8
Scraping page 9
Scraping page 10
Scraping page 11
Scraping page 12
Scraping page 13
Scraping page 14
Scraping page 15
Scraping page 16
Scraping page 17
Scraping page 18
Scraping page 19
Scraping page 20
Scraping page 21
Scraping page 22
Scraping page 23
Scraping page 24
Scraping page 25
Scraping page 26
Scraping page 27
Scraping page 28
Scraping page 29
Scraping page 30
Scraping page 31
Scraping page 32
Scraping page 33
Scraping page 34
Scraping page 35
Scraping page 36
Scraping page 37
Scraping page 38
Scraping page 39
Scraping page 40
Scraping page 41
Scraping page 42
Scraping page 43
Scraping page 44
Scraping page 45
Scraping page 46
Scraping page 47
Scraping page 48
Scraping page 49
Scraping page 50
Scraping page 51
Scraping page 52
Scraping page 53
Scraping page 54
Scraping page 55
Scraping page 56
Scraping page 57
Scraping page 58
Scraping page 59
Scrapi

In [18]:
df.shape

(10, 1)

## Task 2: Vocabulary Creation
Build vocabulary manually or using sklearn. Print vocabulary size and analyze top frequent words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Using CountVectorizer to build vocabulary easily
vocab_vectorizer = CountVectorizer()
vocab_matrix = vocab_vectorizer.fit_transform(df['clean_text'])

# Print vocabulary size
vocab = vocab_vectorizer.vocabulary_
print(f"Vocabulary Size: {len(vocab)} unique words")

# Analyze top frequent words
word_counts = vocab_matrix.sum(axis=0).A1
vocab_words = vocab_vectorizer.get_feature_names_out()

word_freq = pd.DataFrame({'Word': vocab_words, 'Frequency': word_counts})
word_freq = word_freq.sort_values(by='Frequency', ascending=False)
print("\nTop 10 most frequent words:")
print(word_freq.head(10).to_string(index=False))


## Task 3: Feature Engineering
1. One Hot Encoding (document-level vector)
2. Bag of Words using CountVectorizer
3. TF-IDF using TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df['clean_text'].tolist()

# 1. One Hot Encoding (Binary CountVectorizer)
ohe_vectorizer = CountVectorizer(binary=True)
ohe_matrix = ohe_vectorizer.fit_transform(corpus)

# 2. Bag of Words (CountVectorizer standard)
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(corpus)

# 3. TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("Matrices Generated Successfully.\n")
print("OHE Matrix shape: ", ohe_matrix.shape)
print("BoW Matrix shape: ", bow_matrix.shape)
print("TF-IDF Matrix shape:", tfidf_matrix.shape)


## Task 4: Comparison Analysis & Task 5: Sparse Matrix Analysis

### Task 4 Comparison
| Feature | Represents | Weighting | Use Case |
|---------|------------|-----------|----------|
| **OHE** | Presence/Absence of word | Binary (0 or 1) | Simple tracking of vocabulary. |
| **BoW** | Frequency of word | Integer count | Topic modeling, naive bayes, when count matters. |
| **TF-IDF**| Importance of word | Decimal (Scale of importance) | Search engines, text classification where rare words distinguish meaning. |

**Important words in TF-IDF:** Words that appear frequently in a *single* document but rarely across the *entire corpus* receive the highest weight. 
**Why common words receive lower weight:** Common words (like "good", "the") appear in almost every document, so their Document Frequency (DF) is high. TF-IDF penalizes this, reducing their weight to near zero so the model focuses on unique, differentiating words.

### Task 5 Sparse Matrix Analysis
Sparse matrices are matrices where the vast majority of elements are zero.
*Why are they inefficient?* Storing a massive grid of mostly zeros wastes RAM and compute time. Instead, tools like Scipy use compressed formats (CSR) to store only the non-zero values and their coordinates, saving exponential amounts of memory.

In [ ]:
# Calculate and print Sparsity
def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    zero_elements = total_elements - matrix.nnz
    sparsity = (zero_elements / total_elements) * 100
    return sparsity

print(f"OHE Matrix Sparsity: {calculate_sparsity(ohe_matrix):.2f}%")
print(f"BoW Matrix Sparsity: {calculate_sparsity(bow_matrix):.2f}%")
print(f"TF-IDF Matrix Sparsity: {calculate_sparsity(tfidf_matrix):.2f}%")


## Task 6: Real-world Questions
1. **Why BoW fails in semantic meaning:** BoW completely ignores word order and context. For example, "Not good, very bad" and "Not bad, very good" have the exact same BoW vector but opposite meanings (sarcasm and negation are lost). It also treats synonyms as completely different features instead of relating them.
2. **When to use BoW vs TF-IDF:** 
   - *BoW:* Useful in basic spam detection filters, or short documents where simply mentioning a keyword is a strong indicator (e.g., ticket routing by exact keyword match).
   - *TF-IDF:* Preferred in search engine ranking algorithms, document clustering, and sentiment analysis pipeline where context-distinguishing words carry the signal.
3. **Limitations of TF-IDF:** 
   - Still suffers from the "Curse of Dimensionality" (large vocab size).
   - Cannot capture OOV (Out of Vocabulary) semantics.
   - Does not capture positional information or deep context (which modern LLMs/Word Embeddings like Word2Vec/BERT solve).

## Task 7: Mini Use Case (Sentiment Classification)
1. Perform sentiment classification (positive vs negative reviews)
2. Use Logistic Regression or Naive Bayes
3. Compare performance using BoW and TF-IDF features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_bow = bow_matrix
X_tfidf = tfidf_matrix
y = df['sentiment']

# Split data
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tf, X_test_tf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Logistic Regression on BoW
model_bow = LogisticRegression()
model_bow.fit(X_train_bow, y_train)
y_pred_bow = model_bow.predict(X_test_bow)
acc_bow = accuracy_score(y_test, y_pred_bow)

# Logistic Regression on TF-IDF
model_tf = LogisticRegression()
model_tf.fit(X_train_tf, y_train)
y_pred_tf = model_tf.predict(X_test_tf)
acc_tf = accuracy_score(y_test, y_pred_tf)

print("=== Classification Model Accuracy ===")
print(f"Bag of Words (BoW) Accuracy: {acc_bow * 100:.2f}%")
print(f"TF-IDF Feature Accuracy:     {acc_tf * 100:.2f}%")

print("\nObservation: TF-IDF often provides slightly better or identical performance to BoW on normalized context text, by suppressing noisy common words. Depending on the dataset length, BoW might overfit to common positive/negative words.")
